In [ ]:
from torchvision import transforms
from torch.utils.data import DataLoader, Dataset, Subset
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
import pandas as pd
import pickle
import os
from PIL import Image
from matplotlib.pyplot import GridSpec
from sklearn.preprocessing import LabelEncoder

In [ ]:
DATA_DIR = "../../data"
DATASET_DIR = f"{DATA_DIR}/processed"

In [ ]:
train_df = pd.read_csv(f"{DATASET_DIR}/train1_multi.csv")
val_df = pd.read_csv(f"{DATASET_DIR}/val1_multi.csv")
test_df = pd.read_csv(f"{DATASET_DIR}/test1_multi.csv")

In [ ]:
# shuffle val and test
val_df = val_df.sample(frac=1, random_state=42).reset_index(drop=True)
test_df = test_df.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
train_df["Label"].value_counts()

In [ ]:
# verify the images in train are not in val or test
train_images = set(train_df["Image"])
val_images = set(val_df["Image"])
test_images = set(test_df["Image"])

assert len(train_images.intersection(val_images)) == 0
assert len(train_images.intersection(test_images)) == 0

# verify the images in val are not in test
assert len(val_images.intersection(test_images)) == 0

In [ ]:
class CustomDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None, epsilon=0):
        self.dataframe = dataframe
        self.image_dir = image_dir
        self.transform = transform
        self.epsilon = epsilon
    
    def __len__(self):
        return len(self.dataframe)
    
    def __getitem__(self, idx):
        image_path = os.path.join(self.image_dir, self.dataframe.iloc[idx]["Image"])
        label = torch.as_tensor(self.dataframe.iloc[idx]["Label"], dtype=torch.long)
        
        # Load and binarize image
        with Image.open(image_path) as img:
            if self.transform:
                img = self.transform(img)  # Shape: [1, H, W], values 0 or 1
        
        # Generate perturbation mask
        H, W = img.shape[1], img.shape[2]
        perturb_mask = torch.zeros((H, W), dtype=torch.bool)
        if self.epsilon > 0:
            # Randomly select `epsilon` pixels to perturb
            flat_indices = torch.randperm(H * W)[:self.epsilon]
            perturb_mask.view(-1)[flat_indices] = True
        
        # Create interval bounds (lower and upper channels)
        lower = img.squeeze(0).clone()  # Shape: [H, W]
        upper = img.squeeze(0).clone()
        lower[perturb_mask] = 0  # Perturbed pixels: lower bound = 0
        upper[perturb_mask] = 1  # Perturbed pixels: upper bound = 1
        
        # Stack into 2 channels (lower + upper bounds)
        interval_img = torch.stack([lower, upper], dim=0)  # Shape: [2, H, W]
        
        return interval_img, label

In [ ]:
LE = LabelEncoder()
LE.fit(train_df["Label"].unique())

# LE.classes_

# # swap classes in the label encoder
# swapped_classes = LE.classes_.copy()
# swapped_classes[0], swapped_classes[1] = swapped_classes[1], swapped_classes[0]

# LE.classes_ = swapped_classes

train_df_encoded = train_df.copy()
train_df_encoded["Label"] = LE.transform(train_df_encoded["Label"])

val_df_encoded = val_df.copy()
val_df_encoded["Label"] = LE.transform(val_df_encoded["Label"])

train_df_encoded.head()

In [ ]:
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.ToTensor(),
])

train_dataset = CustomDataset(train_df_encoded, f"{DATASET_DIR}/images", transform=transform, epsilon=3)
val_dataset = CustomDataset(val_df_encoded, f"{DATASET_DIR}/images", transform=transform)

In [ ]:
from torch.utils.data import WeightedRandomSampler
from sklearn.utils.class_weight import compute_class_weight

labels = train_df_encoded["Label"].values
class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(labels), y=labels)
sample_weights = class_weights[labels]

sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

In [ ]:
BATCH_SIZE = 40
train_data_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_data_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
class BinaryAbstractionCNN(nn.Module):
    def __init__(self, img_size=32, num_classes=2):
        super().__init__()
        # Input: 2 channels (lower/upper bounds for binary pixels)
        self.features = nn.Sequential(
            nn.Conv2d(2, 32, kernel_size=3, padding=1),  # 2 input channels
            nn.ReLU(),
            nn.MaxPool2d(2),  # e.g., 28x28 → 14x14
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 14x14 → 7x7
        )
        # Calculate flattened size based on img_size
        self.flattened_size = 64 * (img_size // 4) * (img_size // 4)
        self.classifier = nn.Sequential(
            nn.Linear(self.flattened_size, 64),
            nn.ReLU(),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

In [ ]:
dummy = torch.zeros(40, 2, 32, 32, dtype=torch.float32)
model = BinaryAbstractionCNN()
model(dummy).shape

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
criterion_cross_entropy = nn.CrossEntropyLoss
criterion_logits = nn.BCEWithLogitsLoss

In [ ]:
# calculate metrics for binary classification
# accuracy, precision, recall, f1 score
# use sklearn to calculate these metrics
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import confusion_matrix

def calculate_metrics(y_true, y_pred):
    accuracy = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average="macro")
    recall = recall_score(y_true, y_pred, average="macro")
    f1 = f1_score(y_true, y_pred, average="macro")
    
    return {"accuracy": accuracy, "precision": precision, "recall": recall, "f1": f1}

In [ ]:
def train_model_reduceonplateau(model_class, loss_class, train_dataloader, val_dataloader, num_epochs=50, lr=1e-3, early_stop=10, device="cuda", weights=None, pos_weights=None, model_savepath=None, weight_decay=0.0, debug=False):
    model = model_class(num_classes=6).to(device)
    
    if loss_class == nn.BCEWithLogitsLoss:
        if pos_weights is not None:
            criterion = loss_class(pos_weight=pos_weights)
        elif weights is not None:
            criterion = loss_class(weight=weights)
        else:
            criterion = loss_class()
    elif loss_class == nn.CrossEntropyLoss:
        if weights is not None:
            criterion = loss_class(weight=weights)
        else:
            criterion = loss_class()
    else:
        criterion = loss_class()

    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay, betas=(0.9, 0.999))
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.1)

    best_val_loss = np.inf
    best_epoch = 0
    best_model = None
    early_stop_counter = 0

    history = {
        "train_loss": [], 
        "val_loss": [],
        "train_metrics": {
            "accuracy": [], 
            "precision": [], 
            "recall": [], 
            "f1": []
        },
        "val_metrics": {
            "accuracy": [], 
            "precision": [], 
            "recall": [], 
            "f1": []
        }
    }
    
    for epoch in range(num_epochs):
        print(f"\nEpoch {epoch+1}/{num_epochs}")
        print("-" * 50)

        # Training phase
        model.train()
        train_loss = 0.0
        train_preds = []
        train_true = []

        for images, labels in tqdm(train_dataloader, desc="Training Batches"):
            if loss_class == nn.CrossEntropyLoss:
                labels = labels.long().to(device)
            else:
                labels = labels.float().to(device)
            
            images = images.to(device)
            
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

            if loss_class == nn.CrossEntropyLoss:
                preds = torch.argmax(outputs, dim=1).detach().cpu().numpy()
            else:
                probs = torch.sigmoid(outputs).detach().cpu().numpy()
                preds = np.argmax(probs, axis=1)

            true = labels.detach().cpu().numpy()
            train_preds.extend(preds)
            train_true.extend(true)

        train_loss /= len(train_dataloader)
        train_metrics = calculate_metrics(train_true, train_preds)

        # Validation phase
        val_loss = 0.0
        val_preds = []
        val_true = []

        with torch.no_grad():
            model.eval()
            for images, labels in tqdm(val_dataloader, desc="Validation Batches"):
                if loss_class == nn.CrossEntropyLoss:
                    labels = labels.long().to(device)
                else:
                    labels = labels.float().to(device)

                images = images.to(device)
                
                outputs = model(images)
                loss = criterion(outputs, labels)
                val_loss += loss.item()

                if loss_class == nn.CrossEntropyLoss:
                    preds = torch.argmax(outputs, dim=1).detach().cpu().numpy()
                else:
                    probs = torch.sigmoid(outputs).detach().cpu().numpy()
                    preds = np.argmax(probs, axis=1)

                true = labels.detach().cpu().numpy()
                val_preds.extend(preds)
                val_true.extend(true)

        val_loss /= len(val_dataloader)
        val_metrics = calculate_metrics(val_true, val_preds)

        # Update history
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        
        for metric in ["accuracy", "precision", "recall", "f1"]:
            history["train_metrics"][metric].append(train_metrics[metric])
            history["val_metrics"][metric].append(val_metrics[metric])

        if debug:
            print(f"\nLoss - Train: {train_loss:.4f}, Val: {val_loss:.4f}")
            print("\nTraining Metrics:")
            print(f"Accuracy: {train_metrics['accuracy']:.4f}, Precision: {train_metrics['precision']:.4f}, Recall: {train_metrics['recall']:.4f}, F1: {train_metrics['f1']:.4f}")
            print("\nValidation Metrics:")
            print(f"Accuracy: {val_metrics['accuracy']:.4f}, Precision: {val_metrics['precision']:.4f}, Recall: {val_metrics['recall']:.4f}, F1: {val_metrics['f1']:.4f}")
        else:
            print(f"\nLoss - Train: {train_loss:.4f}, Val: {val_loss:.4f}")
            print(f"Accuracy - Train: {train_metrics['accuracy']:.4f}, Val: {val_metrics['accuracy']:.4f}")

        # Early stopping with model saving
        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch
            best_model = model.state_dict()
            if model_savepath:
                torch.save({
                    'epoch': epoch,
                    'model_state_dict': model.state_dict(),
                    'optimizer_state_dict': optimizer.state_dict(),
                    'loss': val_loss,
                    'scheduler_state_dict': scheduler.state_dict(),
                    'learning_rate': scheduler.get_last_lr(),
                    'history': history
                }, model_savepath)
            early_stop_counter = 0
        else:
            early_stop_counter += 1
            if early_stop_counter >= early_stop:
                print(f"\nEarly stopping at epoch {epoch+1}")
                break

        scheduler.step(val_loss)

    print(f"\nBest model was from epoch {best_epoch+1} with validation loss {best_val_loss:.4f}")
    return best_model, history

In [ ]:
def plot_metrics(history):
    gs = GridSpec(3, 2)
    fig = plt.figure(figsize=(12, 12))

    ax = [
        [fig.add_subplot(gs[0, :])],
        [fig.add_subplot(gs[1, 0]), fig.add_subplot(gs[1, 1])],
        [fig.add_subplot(gs[2, 0]), fig.add_subplot(gs[
            2, 1])]
    ]

    # Loss
    ax[0][0].plot(history["train_loss"], label="Train Loss")
    ax[0][0].plot(history["val_loss"], label="Val Loss")
    ax[0][0].set_title("Loss")
    ax[0][0].legend()

    # Accuracy
    ax[1][0].plot(history["train_metrics"]["accuracy"], label="Train Accuracy")
    ax[1][0].plot(history["val_metrics"]["accuracy"], label="Val Accuracy")
    ax[1][0].set_title("Accuracy")
    ax[1][0].legend()

    # Precision
    ax[1][1].plot(history["train_metrics"]["precision"], label="Train Precision")
    ax[1][1].plot(history["val_metrics"]["precision"], label="Val Precision")
    ax[1][1].set_title("Precision")
    ax[1][1].legend()

    # Recall
    ax[2][0].plot(history["train_metrics"]["recall"], label="Train Recall")
    ax[2][0].plot(history["val_metrics"]["recall"], label="Val Recall")
    ax[2][0].set_title("Recall")
    ax[2][0].legend()

    # F1
    ax[2][1].plot(history["train_metrics"]["f1"], label="Train F1")
    ax[2][1].plot(history["val_metrics"]["f1"], label="Val F1")
    ax[2][1].set_title("F1")
    ax[2][1].legend()

    plt.tight_layout()
    plt.show()


In [ ]:
class_weights = compute_class_weight(class_weight="balanced", classes=np.unique(train_df_encoded["Label"]), y=train_df_encoded["Label"])
class_weights = torch.as_tensor(class_weights, dtype=torch.float32).to(device)

class_weights

In [ ]:
model_savepath = f"../../models/checkpoints/multi_cnn/"
os.makedirs(model_savepath, exist_ok=True)

In [ ]:
best_model, history = train_model_reduceonplateau(BinaryAbstractionCNN, criterion_cross_entropy, train_data_loader, val_data_loader, num_epochs=50, lr=1e-3, early_stop=10, device=device, weights=None, model_savepath=f"{model_savepath}abstract_cnn60k.pt", debug=False)

In [ ]:
plot_metrics(history)


In [ ]:
test_data_encoded = test_df.copy()
test_data_encoded["Label"] = LE.transform(test_data_encoded["Label"])

test_dataset = CustomDataset(test_data_encoded, f"{DATASET_DIR}/images", transform=transform)
test_data_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
from sklearn.metrics import confusion_matrix

def test_model(model, test_loader, device="cuda"):  # Changed to test_loader for clarity
    model.eval()
    preds = []
    true = []
    
    with torch.no_grad():
        for images, labels in tqdm(test_loader, desc="Testing Batches"):
            images = images.to(device)
            labels = labels.to(device)  # Keep as long tensor
            
            outputs = model(images)
            batch_preds = torch.argmax(outputs, dim=1)
            
            preds.extend(batch_preds.cpu().numpy())
            true.extend(labels.cpu().numpy())  # Labels should already be integers

    return true, preds

def test_metrics(true, preds):
    metrics = calculate_metrics(true, preds)

    accuracy = metrics["accuracy"]
    precision = metrics["precision"]
    recall = metrics["recall"]
    f1 = metrics["f1"]

    
    # confusion matrix
    cm = confusion_matrix(true, preds, normalize=None)

    # plot confusion matrix
    plt.figure(figsize=(8, 6))
    plt.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)

    plt.title("Confusion Matrix")

    plt.colorbar()

    # 1 is benign, 0 is malicious in predictions and labels

    tick_marks = np.arange(len(LE.classes_))
    plt.xticks(tick_marks, LE.classes_)
    plt.yticks(tick_marks, LE.classes_)
    plt.tight_layout()


    for i in range(len(LE.classes_)):
        for j in range(len(LE.classes_)):
            plt.text(j, i, cm[i, j], horizontalalignment="center", color="white" if cm[i, j] > cm.max() / 2 else "black")

    plt.xlabel("Predicted")
    plt.ylabel("Actual")



    plt.show()

    print(f"Accuracy: {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")

    return accuracy, precision, recall, f1


In [ ]:
# sample 12000 data points from test data with equal distribution of benign and malicious samples
test_data_sample = test_data_encoded.groupby("Label").sample(2000, random_state=42)

# shuffle the data
test_data_sample = test_data_sample.sample(frac=1, random_state=42).reset_index(drop=True) 


test_data_sample["Label"].value_counts()

test_dataset_sample = CustomDataset(test_data_sample, f"{DATASET_DIR}/images", transform=transform)
test_data_loader_sample = DataLoader(test_dataset_sample, batch_size=BATCH_SIZE, shuffle=False)

In [ ]:
model = BinaryAbstractionCNN(num_classes=6).to(device)
checkpoint = torch.load(f"{model_savepath}abstract_cnn60k.pt", weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])


In [ ]:
test_model_sample = test_model(model, test_data_loader_sample, device=device)
test_metrics_sample = test_metrics(*test_model_sample)